# Multimodal Model Training

This notebook trains a multimodal model that combines image and clinical data for ulcer prediction.

## Import Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

print(f"PyTorch version: {torch.__version__}")
print(f"GPU Available: {torch.cuda.is_available()}")

## Load Clinical Data

In [ ]:
# Load clinical data
try:
    clinical_df = pd.read_csv("datasets/clinical_data/patient_data.csv")
    print(f"Clinical data shape: {clinical_df.shape}")
    print(clinical_df.head())
except FileNotFoundError:
    print("Clinical data file not found. Creating sample data...")
    clinical_df = pd.DataFrame({
        'patient_id': range(1, 101),
        'age': np.random.randint(20, 80, 100),
        'bmi': np.random.uniform(18, 40, 100),
        'hba1c': np.random.uniform(6, 12, 100)
    })

## Define Multimodal Dataset

In [ ]:
class MultimodalDataset(Dataset):
    """Dataset combining image and clinical data"""
    
    def __init__(self, image_dir: str, clinical_data: pd.DataFrame):
        self.image_dir = Path(image_dir)
        self.clinical_data = clinical_data
        self.images = list(self.image_dir.glob("*.jpg")) + list(self.image_dir.glob("*.png"))
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        # Placeholder for actual image loading
        image = torch.randn(3, 256, 256)
        
        # Get clinical features
        clinical = torch.randn(10)  # 10 clinical features
        
        # Label (0 or 1)
        label = torch.tensor([np.random.randint(0, 2)], dtype=torch.float32)
        
        return image, clinical, label

# Initialize dataset
dataset = MultimodalDataset("datasets/images/ulcers", clinical_df)
print(f"Dataset size: {len(dataset)}")

## Define Multimodal Architecture

In [ ]:
class MultimodalModel(nn.Module):
    def __init__(self, num_clinical_features=10, num_classes=1):
        super().__init__()
        
        # Image branch
        self.image_branch = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten()
        )
        
        # Clinical branch
        self.clinical_branch = nn.Sequential(
            nn.Linear(num_clinical_features, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.BatchNorm1d(32)
        )
        
        # Fusion layer
        self.fusion = nn.Sequential(
            nn.Linear(128 + 32, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64, num_classes)
        )
    
    def forward(self, image, clinical):
        img_feat = self.image_branch(image)
        clin_feat = self.clinical_branch(clinical)
        fused = torch.cat([img_feat, clin_feat], dim=1)
        output = torch.sigmoid(self.fusion(fused))
        return output

model = MultimodalModel(10, 1)
print(model)

## Training Setup

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

print(f"Training on: {device}")

## Train Model

In [ ]:
epochs = 5
losses = []

for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    
    for batch_idx, (images, clinical, labels) in enumerate(dataloader):
        images = images.to(device)
        clinical = clinical.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images, clinical)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(dataloader)
    losses.append(avg_loss)
    print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f}")

print("Training completed!")

## Plot Results

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(losses, marker='o')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Multimodal Model Training Loss')
plt.grid(True)
plt.show()

## Save Model

In [ ]:
Path("model_weights").mkdir(exist_ok=True)
torch.save(model.state_dict(), "model_weights/multimodal_model.pth")
print("Model saved to model_weights/multimodal_model.pth")